# NYC Hotel Cache Refresh Optimization

**Goal:** Find the optimal cache refresh strategy to minimise stale hotel prices.

**Staleness:** `|cached_price - true_price| / cached_price > STALENESS_THRESHOLD` where `STALENESS_THRESHOLD=0.01` means 1% relative error.

**Sections:**
1. Setup & Configuration
2. Data Loading & Flattening
3. Basic EDA
4. Cache Key Decision (R-squared test)
5. Price Time-Series Construction
6. Hypothesis Tests
7. Adaptive Bucket Discovery
8. Sparsity & Poisson Diagnostics
9. Kaplan-Meier Survival Analysis
10. Fixed TTL Grid Search
11. Poisson Process TTL (conditional)
12. Poisson Regression GLM (conditional)
13. Strategy Simulation
14. Synthetic Data Validation
15. Presentation Figures


## Section 0 — Setup & Configuration

All tunable parameters live here. Change any value and re-run the notebook.

In [ ]:
import subprocess, sys

def _install(pkg, import_as=None):
    try: __import__(import_as or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "--user", "-q"])

_install("lifelines")
_install("scikit-learn", "sklearn")
_install("statsmodels")

import pandas as pd
import numpy as np
import glob, os, warnings
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
})
print("Libraries loaded.")

In [ ]:
# ─────────────────────────────────────────────────────
# CONFIGURABLE PARAMETERS  (edit here, then re-run)
# ─────────────────────────────────────────────────────
DATA_DIR_CANDIDATES = [
    r"C:\Users\sriva\Downloads\NYC hotel data",
    "/Volumes/Backup Plus/Mac Air/ORIE 2026/Sabre Project/LLM Intent Cache/raw_data/nyc_hotel_data",
    os.path.abspath("../LLM Intent Cache/raw_data/nyc_hotel_data"),
]
DATA_DIR = next((d for d in DATA_DIR_CANDIDATES if os.path.isdir(d)), DATA_DIR_CANDIDATES[0])

STALENESS_THRESHOLD      = 0.01   # fraction — 1% relative price change
TARGET_FRESHNESS         = 0.80   # target P(cache still fresh)
MIN_TTL                  = 0.25   # hours
MAX_TTL                  = 336.0  # hours (2 weeks)

# Fixed-TTL grid to search (hours)
TTL_GRID = [0.25, 0.5, 0.75, 1, 1.5, 2, 2.5, 3] + list(range(4, 25))

TOP_N_CHAINS             = 10     # top N chains by volume; rest -> 'Other'
SIMULATION_SAMPLE_ROWS   = 500_000

OVERDISPERSION_THRESHOLD = 2.0    # Var/Mean ratio; > this -> Poisson suspect
ZERO_INFLATION_THRESHOLD = 0.30   # fraction of groups with 0 changes
MIN_EVENTS_PER_KM_GROUP  = 30     # min change events for a reliable KM curve
LOGRANK_SIGNIFICANCE     = 0.05

# Cache size constraints for fixed-TTL selection
MAX_CACHE_SIZE_MB        = 60
AVG_ENTRY_SIZE_BYTES     = 500
MIN_CACHE_UTIL_FRACTION  = 0.20
MIN_CACHE_UTIL_MB        = MAX_CACHE_SIZE_MB * MIN_CACHE_UTIL_FRACTION

# R-squared threshold for horizon-in-key decision.
# R^2 measures how much horizon EXPLAINS price level (0=none, 1=perfect).
# HIGH R^2 (>threshold) -> horizon predicts price level -> investigate as key component
# LOW  R^2 (<threshold) -> horizon only affects volatility -> belongs in TTL, NOT key
R2_HORIZON_KEY_THRESHOLD = 0.30
MIN_OBS_FOR_R2           = 3      # min obs per key to include in R^2 analysis

MAX_HORIZON_BUCKETS      = 8
MAX_DURATION_BUCKETS     = 6

print("Configuration loaded.")
print(f"  TARGET_FRESHNESS        = {TARGET_FRESHNESS}")
print(f"  STALENESS_THRESHOLD     = {STALENESS_THRESHOLD:.2%} (relative)")
print(f"  MIN_TTL / MAX_TTL       = {MIN_TTL}h / {MAX_TTL}h")
print(f"  TTL_GRID                = {TTL_GRID}")
print(f"  MAX_CACHE_SIZE_MB       = {MAX_CACHE_SIZE_MB}")
print(f"  AVG_ENTRY_SIZE_BYTES    = {AVG_ENTRY_SIZE_BYTES}")
print(f"  MIN_CACHE_UTIL_FRACTION = {MIN_CACHE_UTIL_FRACTION:.0%}")
print(f"  MIN_CACHE_UTIL_MB       = {MIN_CACHE_UTIL_MB}")
print(f"  DATA_DIR                = {DATA_DIR}")


## Section 1 — Data Loading & Flattening

**What we do:**
- Load all 100 parquet files
- Explode `convertedrate_infos` (numpy array of dicts) so each rate entry becomes its own row
- Extract `amount_after_tax`, `currency_code`, `rate_source`
- Filter to USD rows
- Compute derived columns: `duration_nights`, `horizon_days`, `stay_start_month`

**Why explode?** One hotel response carries multiple rates (e.g. rate_source=100 at $2,093 AND rate_source=110 at $3,602). Each is a distinct cached product with its own price and potentially different volatility.

In [ ]:
def load_and_flatten(filepath):
    """Load one parquet shard, explode convertedrate_infos -> one row per rate."""
    df = pd.read_parquet(filepath)
    df["rq_timestamp"]       = pd.to_datetime(df["rq_timestamp"], utc=True)
    df["rq_stay_start_date"] = pd.to_datetime(df["rq_stay_start_date"], format="ISO8601")
    df["rq_stay_end_date"]   = pd.to_datetime(df["rq_stay_end_date"],   format="ISO8601")

    keep = ["rq_correlation_id", "rq_timestamp", "hotel_code", "chain_code",
            "rq_stay_start_date", "rq_stay_end_date",
            "location_city_code", "sabre_rating", "convertedrate_infos"]
    df = df[keep].copy()

    # convertedrate_infos is a numpy array of dicts -> convert to Python list
    def safe_to_list(x):
        if x is None:                 return []
        if isinstance(x, np.ndarray): return x.tolist()  # preserves inner dicts
        if isinstance(x, list):       return x
        return []

    df["convertedrate_infos"] = df["convertedrate_infos"].apply(safe_to_list)
    df = df[df["convertedrate_infos"].apply(len) > 0]

    # Explode: one row per rate entry
    df = df.explode("convertedrate_infos").reset_index(drop=True)
    df = df[df["convertedrate_infos"].apply(lambda x: isinstance(x, dict))]

    df["amount_after_tax"] = df["convertedrate_infos"].apply(lambda x: x.get("amount_after_tax"))
    df["currency_code"]    = df["convertedrate_infos"].apply(lambda x: x.get("currency_code", ""))
    df["rate_source"]      = df["convertedrate_infos"].apply(lambda x: x.get("rate_source", ""))
    df = df.drop(columns=["convertedrate_infos"])
    df = df[df["amount_after_tax"].notna()]
    return df


files = sorted(glob.glob(os.path.join(DATA_DIR, "get_hotel_avail_*.parquet")))
print(f"Found {len(files)} parquet files. Loading...")
parts = [load_and_flatten(f) for f in files]
raw   = pd.concat(parts, ignore_index=True)
print(f"Total rows after explode : {raw.shape[0]:,}")
print(f"Columns: {raw.columns.tolist()}")
raw.head(3)

In [ ]:
# ── Filter USD, compute derived fields ────────────────────────────────────
df = raw[raw["currency_code"] == "USD"].copy()
print(f"USD rows : {df.shape[0]:,}  ({df.shape[0]/raw.shape[0]*100:.1f}% of total)")

df["duration_nights"]  = (df["rq_stay_end_date"] - df["rq_stay_start_date"]).dt.days
df["horizon_days"]     = (
    (df["rq_stay_start_date"] - df["rq_timestamp"].dt.tz_convert(None).dt.normalize()).dt.days
).clip(lower=0)  # clip negatives (retrospective searches)
df["stay_start_month"] = df["rq_stay_start_date"].dt.month

# Group chains: top N by volume, rest -> 'Other'
top_chains = df["chain_code"].value_counts().head(TOP_N_CHAINS).index.tolist()
df["chain_grouped"] = df["chain_code"].where(df["chain_code"].isin(top_chains), other="Other")

# Tentative cache key (horizon decision confirmed in Section 3)
# Key = city | hotel | stay_start | duration | rate_source
df["cache_key"] = (
    df["location_city_code"] + "|" +
    df["hotel_code"]         + "|" +
    df["rq_stay_start_date"].dt.strftime("%Y-%m-%d") + "|" +
    df["duration_nights"].astype(str) + "|" +
    df["rate_source"]
)

print(f"Unique cache keys  : {df.cache_key.nunique():,}")
print(f"Stay date range    : {df.rq_stay_start_date.min().date()} -> {df.rq_stay_start_date.max().date()}")
print(f"Horizon range      : {df.horizon_days.min()} - {df.horizon_days.max()} days")
print(f"Duration range     : {df.duration_nights.min()} - {df.duration_nights.max()} nights")
print(f"Rate sources       : {sorted(df.rate_source.unique())}")

## Section 2 — Basic EDA

Understand the distribution of queries, prices, chains, durations and rate sources.

In [ ]:
print("=== Dataset overview ===")
print(f"Request window     : {df.rq_timestamp.min()} -> {df.rq_timestamp.max()}")
print(f"Unique hotels      : {df.hotel_code.nunique():,}")
print(f"Unique chains      : {df.chain_code.nunique()}")
print(f"Unique rate sources: {sorted(df.rate_source.unique())}")
print(f"Unique cache keys  : {df.cache_key.nunique():,}")
print(f"\nPrice stats (USD):")
print(df["amount_after_tax"].describe())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))

# Query volume per hour
hourly = df.groupby(df["rq_timestamp"].dt.floor("h")).size()
hourly.plot(ax=axes[0,0], lw=1.2, color="steelblue")
axes[0,0].set_title("Query volume per hour"); axes[0,0].set_xlabel("")
plt.setp(axes[0,0].xaxis.get_majorticklabels(), rotation=25)

# Price distribution
p99 = df["amount_after_tax"].quantile(0.99)
df[df["amount_after_tax"] < p99]["amount_after_tax"]         .plot.hist(bins=60, ax=axes[0,1], color="coral", edgecolor="white")
axes[0,1].set_title("Price distribution (USD, <99th pct)")
axes[0,1].set_xlabel("Price (USD)")

# Top chains
df["chain_code"].value_counts().head(12).plot.bar(
    ax=axes[0,2], color="steelblue", edgecolor="white")
axes[0,2].set_title("Top 12 chains"); axes[0,2].tick_params(axis="x", rotation=30)

# Rate source
df["rate_source"].value_counts().plot.bar(
    ax=axes[1,0], color="coral", edgecolor="white")
axes[1,0].set_title("Rate source distribution"); axes[1,0].tick_params(axis="x", rotation=0)

# Stay duration
df[df["duration_nights"] <= 14]["duration_nights"].value_counts().sort_index()         .plot.bar(ax=axes[1,1], color="steelblue", edgecolor="white")
axes[1,1].set_title("Stay duration (nights, <=14)"); axes[1,1].tick_params(axis="x", rotation=0)

# Horizon distribution
df[df["horizon_days"] <= 180]["horizon_days"].plot.hist(
    bins=60, ax=axes[1,2], color="purple", edgecolor="white", alpha=0.7)
axes[1,2].set_title("Booking horizon (<=180 days)")
axes[1,2].set_xlabel("Days until stay")

plt.suptitle("Basic EDA", fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

## Section 3 — Cache Key Decision: Does Horizon Belong?

**The question:** For the same `(hotel, city, stay_dates, rate_source)`, does the price *level* vary systematically with horizon (days until stay), or do prices change randomly regardless of horizon?

**How R-squared works here:**
We fit `price = b0 + b1 * horizon_days` for each candidate key with >= 3 observations.
- R^2 = 0 -> horizon explains 0% of price variance -> no relationship
- R^2 = 0.80 -> horizon explains 80% of price variance -> very strong relationship
- **HIGH R^2 (> threshold)** -> horizon predicts price level -> consider as key component
- **LOW R^2 (< threshold)** -> horizon does NOT explain price level -> NOT in key; use only in TTL formula

**How to inspect the scatter plots:** Look for a clear upward or downward trend. A flat trend with scattered points = low R^2 = horizon not in key.

**Practical note:** Even if R^2 is high, including horizon as a continuous value in the key would make every query unique (horizon changes daily), destroying cache efficiency. The R^2 test quantifies the price-level variation; the practical decision also factors in cache efficiency.

In [ ]:
# ── R^2 analysis: fit price ~ horizon_days per cache key ─────────────────
r2_rows = []
for key, grp in df.groupby("cache_key"):
    if len(grp) < MIN_OBS_FOR_R2:
        continue
    x = grp["horizon_days"].values.astype(float)
    y = grp["amount_after_tax"].values.astype(float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < MIN_OBS_FOR_R2:
        continue
    if x[mask].std() == 0:  # all horizon values identical — regression undefined
        continue
    slope, intercept, r_val, p_val, _ = stats.linregress(x[mask], y[mask])
    r2_rows.append({
        "cache_key": key, "r2": r_val**2,
        "p_value": p_val, "slope": slope, "n_obs": int(mask.sum())
    })

r2_df = pd.DataFrame(r2_rows)
print(f"Keys analysed (>= {MIN_OBS_FOR_R2} obs): {len(r2_df):,}")
print(f"\nR^2 summary:")
print(r2_df["r2"].describe())
print(f"\nMedian R^2             : {r2_df['r2'].median():.4f}")
print(f"% keys with R^2 > 0.30 : {(r2_df['r2'] > 0.30).mean()*100:.1f}%")
print(f"% keys with p < 0.05   : {(r2_df['p_value'] < 0.05).mean()*100:.1f}%")

In [ ]:
# ── Scatter grid: price vs horizon for 12 sampled keys ───────────────────
sample_keys_r2 = r2_df.sample(min(12, len(r2_df)), random_state=42)["cache_key"].values

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, key in zip(axes.flatten(), sample_keys_r2):
    grp = df[df["cache_key"] == key]
    ax.scatter(grp["horizon_days"], grp["amount_after_tax"], alpha=0.7, s=18, color="steelblue")
    x = grp["horizon_days"].values.astype(float)
    y = grp["amount_after_tax"].values.astype(float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() >= 2:
        sl, ic, _, _, _ = stats.linregress(x[mask], y[mask])
        xr = np.linspace(x.min(), x.max(), 50)
        ax.plot(xr, ic + sl * xr, "r-", lw=1.5, alpha=0.8)
    r2_val = r2_df.loc[r2_df["cache_key"] == key, "r2"].values
    ax.set_title(f"R^2={r2_val[0]:.3f}" if len(r2_val) else "N/A", fontsize=9)
    ax.set_xlabel("Horizon (days)", fontsize=8)
    ax.set_ylabel("Price ($)", fontsize=8)
    ax.tick_params(labelsize=7)

plt.suptitle(
    "Price vs Horizon - 12 Sampled Cache Keys\n"
    "Flat trend + scattered cloud -> low R^2 -> horizon NOT in key\n"
    "Clear slope -> high R^2 -> horizon predicts price level -> investigate further",
    fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── R^2 distribution histogram ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

r2_df["r2"].plot.hist(bins=50, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].axvline(R2_HORIZON_KEY_THRESHOLD, color="red", lw=2, linestyle="--",
                label=f"Threshold = {R2_HORIZON_KEY_THRESHOLD}")
axes[0].axvline(r2_df["r2"].median(), color="orange", lw=2,
                label=f"Median = {r2_df['r2'].median():.3f}")
axes[0].set_title("R^2 distribution across all qualifying keys")
axes[0].set_xlabel("R^2 (horizon explaining price level)")
axes[0].legend()

sorted_r2 = np.sort(r2_df["r2"].values)
cdf = np.linspace(0, 1, len(sorted_r2))
axes[1].plot(sorted_r2, cdf, color="steelblue", lw=2)
axes[1].axvline(R2_HORIZON_KEY_THRESHOLD, color="red", lw=2, linestyle="--",
                label=f"Threshold = {R2_HORIZON_KEY_THRESHOLD}")
axes[1].axvline(r2_df["r2"].median(), color="orange", lw=2,
                label=f"Median = {r2_df['r2'].median():.3f}")
axes[1].set_title("CDF of R^2 values")
axes[1].set_xlabel("R^2"); axes[1].set_ylabel("Cumulative fraction")
axes[1].legend()

plt.tight_layout(); plt.show()

median_r2 = r2_df["r2"].median()
if median_r2 > R2_HORIZON_KEY_THRESHOLD:
    print(f"WARNING: Median R^2 = {median_r2:.3f} > threshold {R2_HORIZON_KEY_THRESHOLD}")
    print("  Horizon has notable explanatory power over price level.")
    print("  Inspect scatter plots carefully before finalising key decision.")
else:
    print(f"OK: Median R^2 = {median_r2:.3f} < threshold {R2_HORIZON_KEY_THRESHOLD}")
    print("  Horizon does NOT meaningfully explain price level.")
    print("  DECISION: horizon NOT in cache key. It will only influence TTL.")
    print("\n  Final cache key: city | hotel_code | stay_start_date | duration_nights | rate_source")

## Section 4 — Price Time-Series Construction

For each cache key we build an ordered sequence of price observations.

- One row = one observation of a cache key at a specific timestamp
- `price_changed` = True if `|price - prev_price| / prev_price > STALENESS_THRESHOLD`
- `obs_gap_h` = hours since the previous observation of this key (used to estimate change rate lambda)


In [ ]:
# If the same (cache_key, timestamp) appears multiple times (same hotel in
# multiple shops at nearly the same second), take the mean price.
price_ts = (
    df.sort_values(["cache_key", "rq_timestamp"])
    .groupby(["cache_key", "rq_timestamp"], sort=False)
    .agg(
        amount_after_tax   = ("amount_after_tax",   "mean"),
        chain_code         = ("chain_code",         "first"),
        chain_grouped      = ("chain_grouped",      "first"),
        rate_source        = ("rate_source",        "first"),
        rq_stay_start_date = ("rq_stay_start_date", "first"),
        duration_nights    = ("duration_nights",    "first"),
        location_city_code = ("location_city_code", "first"),
    )
    .reset_index()
)
price_ts = price_ts.sort_values(["cache_key", "rq_timestamp"])

# horizon_days per observation (changes at each observation timestamp)
price_ts["horizon_days"] = (
    price_ts["rq_stay_start_date"] - price_ts["rq_timestamp"].dt.tz_convert(None).dt.normalize()
).dt.days.clip(lower=0)

# Price change flags
price_ts["prev_price"] = price_ts.groupby("cache_key")["amount_after_tax"].shift(1)

# Old absolute-change reference rate ($0.01) for side-by-side reporting
old_abs_changed = (
    price_ts["prev_price"].notna() &
    (abs(price_ts["amount_after_tax"] - price_ts["prev_price"]) > STALENESS_THRESHOLD)
)

# New relative-change definition (1% when STALENESS_THRESHOLD=0.01)
price_ts["price_changed"] = (
    price_ts["prev_price"].notna() &
    (price_ts["prev_price"] > 0) &
    (abs(price_ts["amount_after_tax"] - price_ts["prev_price"]) / price_ts["prev_price"] > STALENESS_THRESHOLD)
)

price_ts["price_delta"] = price_ts["amount_after_tax"] - price_ts["prev_price"]
price_ts["price_delta_pct"] = price_ts["price_delta"] / price_ts["prev_price"] * 100

# Observation gap in hours (denominator for lambda estimation)
price_ts["obs_gap_h"] = (
    price_ts.groupby("cache_key")["rq_timestamp"].diff().dt.total_seconds() / 3600
)

print(f"Price time-series rows : {price_ts.shape[0]:,}")
print(f"Unique cache keys      : {price_ts.cache_key.nunique():,}")

changes_df = price_ts[price_ts["prev_price"].notna()].copy()
old_rate = old_abs_changed[price_ts["prev_price"].notna()].mean() * 100
new_rate = changes_df["price_changed"].mean() * 100

print()
print(f"Observations with price change : {changes_df.price_changed.sum():,} / {len(changes_df):,}")
print(f"Change rate (old $0.01 absolute): {old_rate:.2f}%")
print(f"Change rate (new 1% relative)   : {new_rate:.2f}%")

changes_only = changes_df[changes_df["price_changed"]].copy()
print()
print("Among changes:")
print(f"  Increases   : {(changes_only.price_delta > 0).mean()*100:.1f}%")
print(f"  Decreases   : {(changes_only.price_delta < 0).mean()*100:.1f}%")
print(f"  Median |$d| : ${changes_only.price_delta.abs().median():.2f}")
print(f"  Median |%d| : {changes_only.price_delta_pct.abs().median():.1f}%")


## Section 5 — Hypothesis Tests

What drives price volatility? The answers inform:
- Which dimensions to stratify KM survival curves on
- Which features to include in the Poisson GLM
- Whether rate_source belongs in the cache key

In [ ]:
def change_rate_by(col, data=None, min_obs=20):
    """Change rate (%) per group of col."""
    if data is None:
        data = changes_df
    g = data.groupby(col).agg(
        n_obs     = ("price_changed", "count"),
        n_changes = ("price_changed", "sum")
    ).reset_index()
    g["change_rate_pct"] = g["n_changes"] / g["n_obs"] * 100
    return g[g["n_obs"] >= min_obs].sort_values(col)

In [ ]:
# ── H1: Horizon -> change rate ─────────────────────────────────────────
changes_df["horizon_days_int"] = changes_df["horizon_days"].astype(int).clip(0, 180)
h1 = change_rate_by("horizon_days_int", min_obs=50)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].bar(h1["horizon_days_int"], h1["change_rate_pct"],
            color="steelblue", width=1.0, edgecolor="none", alpha=0.7)
axes[0].set_title("H1: Price change rate by horizon (days until stay)")
axes[0].set_xlabel("Horizon (days)"); axes[0].set_ylabel("Change rate (%)")
axes[0].set_xlim(-1, 181)

rolled = h1.set_index("horizon_days_int")["change_rate_pct"]                .rolling(7, center=True, min_periods=3).mean()
axes[1].bar(h1["horizon_days_int"], h1["change_rate_pct"],
            color="steelblue", width=1.0, edgecolor="none", alpha=0.3)
axes[1].plot(rolled.index, rolled.values, color="steelblue", lw=2.5, label="7-day rolling avg")
axes[1].set_title("H1 (smoothed): Change rate vs horizon"); axes[1].set_xlim(-1, 181)
axes[1].set_xlabel("Horizon (days)"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── H2: Duration -> change rate ────────────────────────────────────────
changes_df["duration_int"] = changes_df["duration_nights"].clip(1, 21).astype(int)
h2 = change_rate_by("duration_int", min_obs=50)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(h2["duration_int"].astype(str), h2["change_rate_pct"],
       color="coral", edgecolor="white")
ax.set_title("H2: Price change rate by stay duration (nights)")
ax.set_xlabel("Duration (nights)"); ax.set_ylabel("Change rate (%)")
plt.tight_layout(); plt.show()
print(h2[["duration_int","n_obs","n_changes","change_rate_pct"]].to_string(index=False))

In [ ]:
# ── H3: Seasonality -> price AND change rate ───────────────────────────
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
_tmp = changes_df.copy()
_tmp["stay_start_month"] = _tmp["rq_stay_start_date"].dt.month
h3_rate  = change_rate_by("stay_start_month", data=_tmp, min_obs=50)
month_price = df.groupby(df["rq_stay_start_date"].dt.month)["amount_after_tax"].median()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].bar(h3_rate["stay_start_month"].map(month_names), h3_rate["change_rate_pct"],
            color="steelblue", edgecolor="white")
axes[0].set_title("H3a: Change rate by stay start month")
axes[0].tick_params(axis="x", rotation=30)

month_price.plot.bar(ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("H3b: Median price by stay start month")
axes[1].set_xlabel("Month (number)"); axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# ── H4: Rate source -> change rate AND price ───────────────────────────
h4_rate  = change_rate_by("rate_source", min_obs=50)
rs_price = df.groupby("rate_source")["amount_after_tax"].median()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(h4_rate["rate_source"].astype(str), h4_rate["change_rate_pct"],
            color="steelblue", edgecolor="white")
axes[0].set_title("H4a: Change rate by rate_source")
axes[0].set_xlabel("Rate source"); axes[0].set_ylabel("Change rate (%)")

rs_price.plot.bar(ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("H4b: Median price by rate_source")
axes[1].set_xlabel("Rate source"); axes[1].set_ylabel("Median price (USD)")
plt.tight_layout(); plt.show()
print(h4_rate[["rate_source","n_obs","n_changes","change_rate_pct"]].to_string(index=False))

In [ ]:
# ── H5: Chain -> change rate ───────────────────────────────────────────
h5 = change_rate_by("chain_code", min_obs=100).sort_values("change_rate_pct")

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(h5["chain_code"].tail(20).astype(str),
        h5["change_rate_pct"].tail(20), color="steelblue", edgecolor="white")
ax.set_title("H5: Change rate - Top 20 chains")
ax.set_xlabel("Change rate (%)")
plt.tight_layout(); plt.show()

In [ ]:
# ── H6: Interaction heatmap - rate_source x horizon (coarse) ──────────
changes_df["horizon_coarse"] = pd.cut(
    changes_df["horizon_days"], bins=[-1,7,30,90,365],
    labels=["0-7d","8-30d","31-90d","91+d"])
heat_data = (
    changes_df.groupby(["rate_source","horizon_coarse"], observed=True)
    .agg(n_obs=("price_changed","count"), n_changes=("price_changed","sum"))
    .assign(change_rate=lambda x: x["n_changes"]/x["n_obs"]*100)
    .reset_index()
    .pivot(index="rate_source", columns="horizon_coarse", values="change_rate")
)
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(heat_data, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax,
            cbar_kws={"label": "Change rate (%)"})
ax.set_title("H6: Price change rate (%) - rate_source x horizon bucket")
plt.tight_layout(); plt.show()
print(heat_data)

## Section 6 — Adaptive Bucket Discovery

Instead of hardcoding bucket boundaries, we find them from the data using K-Means 1D:

1. Compute change rate per integer horizon value (0, 1, ..., 180)
2. Fit K-Means for k=2 to `MAX_HORIZON_BUCKETS`; record inertia
3. Find optimal k via the elbow (maximum curvature in the inertia curve)
4. Break points = midpoints between consecutive cluster centroids

Same approach for duration. These bucket definitions feed into all downstream sections.

In [ ]:
def discover_breaks_1d(rate_series, max_k, name):
    """
    Find natural break points in a 1-D change-rate series.

    Parameters
    ----------
    rate_series : pd.Series  index=integer value, values=change rate
    max_k       : int        max clusters to try
    name        : str        for labelling plots

    Returns: (breaks, labels, optimal_k)
    """
    idx  = rate_series.dropna().index.values
    vals = rate_series.dropna().values.reshape(-1, 1)
    k_range  = list(range(2, max_k + 1))
    inertias = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        km.fit(vals)
        inertias.append(km.inertia_)

    # Elbow via second derivative
    if len(inertias) >= 3:
        d2       = np.diff(np.diff(inertias))
        opt_idx  = int(np.argmax(d2)) + 2  # +2 because double diff loses 2 elements
        optimal_k = k_range[min(opt_idx, len(k_range) - 1)]
    else:
        optimal_k = k_range[0]

    km_final  = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
    km_final.fit(vals)
    centroids = sorted(km_final.cluster_centers_.flatten())

    # Break points: find where smoothed rate crosses midpoint between consecutive centroids
    breaks = []
    series_smooth = rate_series.rolling(3, center=True, min_periods=1).mean()
    for i in range(len(centroids) - 1):
        mid_rate = (centroids[i] + centroids[i+1]) / 2
        above    = series_smooth[series_smooth >= mid_rate].index
        below    = series_smooth[series_smooth < mid_rate].index
        if len(above) > 0 and len(below) > 0:
            crossing = int(above[-1]) if above[-1] < below[-1] else max(0, int(below[0]) - 1)
            breaks.append(max(0, crossing))
    breaks = sorted(set(breaks))

    # Build bucket labels
    boundaries = [-1] + breaks + [int(idx.max())]
    labels = []
    for i in range(len(boundaries) - 1):
        lo = boundaries[i] + 1
        hi = boundaries[i + 1]
        suffix = "+" if i == len(boundaries) - 2 else ""
        labels.append(f"{lo}-{hi}{suffix}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].bar(idx, rate_series[idx], color="steelblue", width=1, edgecolor="none", alpha=0.6)
    for b in breaks:
        axes[0].axvline(b, color="red", lw=1.5, linestyle="--")
    axes[0].set_title(f"{name} - Change rate with discovered break points (k={optimal_k})")
    axes[0].set_xlabel(name); axes[0].set_ylabel("Change rate (%)")

    axes[1].plot(k_range, inertias, "o-", color="steelblue", lw=2)
    axes[1].axvline(optimal_k, color="red", lw=1.5, linestyle="--", label=f"Optimal k={optimal_k}")
    axes[1].set_title(f"{name} - Inertia vs k (elbow)")
    axes[1].set_xlabel("k"); axes[1].set_ylabel("Inertia"); axes[1].legend()
    plt.tight_layout(); plt.show()

    print(f"{name}: k={optimal_k}, breaks={breaks}")
    print(f"  Labels: {labels}")
    return breaks, labels, optimal_k


print("=== Horizon bucket discovery ===")
HORIZON_BREAKS, HORIZON_LABELS, _ = discover_breaks_1d(
    h1.set_index("horizon_days_int")["change_rate_pct"],
    max_k=MAX_HORIZON_BUCKETS, name="Horizon (days)")

print("\n=== Duration bucket discovery ===")
DURATION_BREAKS, DURATION_LABELS, _ = discover_breaks_1d(
    h2.set_index("duration_int")["change_rate_pct"],
    max_k=MAX_DURATION_BUCKETS, name="Duration (nights)")

In [ ]:
# ── Assign buckets ────────────────────────────────────────────────────
def assign_bucket(value, breaks, labels):
    """Return bucket label for an integer value given break points."""
    for i, b in enumerate(breaks):
        if value <= b:
            return labels[i]
    return labels[-1]

for _df in [price_ts, changes_df]:
    _df["horizon_bucket"] = _df["horizon_days"].apply(
        lambda x: assign_bucket(int(x), HORIZON_BREAKS, HORIZON_LABELS))
    _df["duration_bucket"] = _df["duration_nights"].clip(1).apply(
        lambda x: assign_bucket(int(x), DURATION_BREAKS, DURATION_LABELS))

print("Horizon bucket value counts (price_ts):")
print(price_ts["horizon_bucket"].value_counts().sort_index())
print("\nDuration bucket value counts (price_ts):")
print(price_ts["duration_bucket"].value_counts().sort_index())

## Section 7 — Sparsity & Poisson Diagnostics

Before using any Poisson-based model we verify:
1. **Per-key sparsity** — do individual cache keys have enough observations?
2. **Per-group sparsity** — do groups `(chain, rate_source, horizon_bucket)` have enough data?
3. **Overdispersion** — does Var(changes) ~ Mean(changes) as Poisson requires?
4. **Zero-inflation** — are there too many groups with 0 observed changes?

If diagnostics fail, Sections 11-12 print a warning and are skipped.

In [ ]:
# ── 7.1 Per-key sparsity ─────────────────────────────────────────────
key_obs_counts = price_ts.groupby("cache_key").size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
key_obs_counts.clip(upper=30).plot.hist(
    bins=30, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Observations per cache key (clipped at 30)")
axes[0].set_xlabel("# observations")

sorted_counts = np.sort(key_obs_counts.values)
cdf = np.arange(1, len(sorted_counts)+1) / len(sorted_counts)
axes[1].plot(sorted_counts, cdf, color="steelblue", lw=2)
for v in [1, 2, 5, 10]:
    pct = (key_obs_counts <= v).mean()*100
    axes[1].axvline(v, color="red", lw=1, linestyle="--", alpha=0.6)
    axes[1].text(v+0.2, pct/100 - 0.05, f"<={v}: {pct:.0f}%", fontsize=8, color="red")
axes[1].set_title("CDF: observations per cache key")
axes[1].set_xlabel("# observations"); axes[1].set_ylabel("Fraction of keys"); axes[1].set_xlim(0,30)
plt.tight_layout(); plt.show()

print("Sparsity (per key):")
for v in [1, 2, 5, 10]:
    print(f"  Keys with <= {v:2d} obs: {(key_obs_counts <= v).mean()*100:.1f}%")

In [ ]:
# ── 7.2 Per-group sparsity ────────────────────────────────────────────
group_agg = (
    changes_df
    .groupby(["chain_grouped","rate_source","horizon_bucket"], observed=True)
    .agg(total_obs=("price_changed","count"),
         total_changes=("price_changed","sum"),
         total_obs_h=("obs_gap_h","sum"))
    .reset_index()
)
group_agg["zero_changes"] = (group_agg["total_changes"] == 0).astype(int)
zero_inf_rate = group_agg["zero_changes"].mean()

print(f"Total groups                 : {len(group_agg):,}")
print(f"Groups with 0 changes        : {group_agg.zero_changes.sum():,} ({zero_inf_rate*100:.1f}%)")
print(f"\nObs per group summary:")
print(group_agg["total_obs"].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
group_agg["total_obs"].clip(upper=500).plot.hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Total observations per group"); axes[0].set_xlabel("# obs")
group_agg["total_changes"].clip(upper=50).plot.hist(bins=25, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Total changes per group"); axes[1].set_xlabel("# changes")
plt.tight_layout(); plt.show()

In [ ]:
# ── 7.3 Overdispersion check ─────────────────────────────────────────
key_group = (
    changes_df
    .groupby(["cache_key","chain_grouped","rate_source","horizon_bucket"], observed=True)
    ["price_changed"].sum().reset_index(name="n_changes_per_key")
)
group_disp = (
    key_group
    .groupby(["chain_grouped","rate_source","horizon_bucket"], observed=True)
    ["n_changes_per_key"].agg(["mean","var","count"]).reset_index()
    .rename(columns={"mean":"mean_ch","var":"var_ch","count":"n_keys"})
)
group_disp = group_disp[(group_disp["n_keys"] >= 5) & (group_disp["mean_ch"] > 0)].copy()
group_disp["dispersion"] = group_disp["var_ch"] / group_disp["mean_ch"]
overall_dispersion = group_disp["dispersion"].median()

print(f"Median Var/Mean (dispersion) : {overall_dispersion:.2f}")
print(f"Groups with disp > {OVERDISPERSION_THRESHOLD}  : {(group_disp.dispersion > OVERDISPERSION_THRESHOLD).mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(group_disp["mean_ch"], group_disp["var_ch"], alpha=0.4, s=20, color="steelblue")
mx = group_disp[["mean_ch","var_ch"]].max().max()
ax.plot([0,mx],[0,mx],"g-",lw=2, label="Var = Mean (Poisson)")
ax.plot([0,mx],[0,mx*OVERDISPERSION_THRESHOLD],"r--",lw=1.5, label=f"Var = {OVERDISPERSION_THRESHOLD}x Mean (threshold)")
ax.set_title("Overdispersion: Var vs Mean of changes per group")
ax.set_xlabel("Mean changes"); ax.set_ylabel("Var changes")
ax.legend(); ax.set_xlim(0); ax.set_ylim(0)
plt.tight_layout(); plt.show()

In [ ]:
# ── 7.4 Diagnostic conclusion ─────────────────────────────────────────
POISSON_VIABLE   = True
diagnostic_notes = []

if overall_dispersion > OVERDISPERSION_THRESHOLD:
    diagnostic_notes.append(
        f"WARNING Overdispersion: median Var/Mean = {overall_dispersion:.2f} "
        f"(threshold {OVERDISPERSION_THRESHOLD}). "
        f"Standard Poisson underestimates variance. "
        f"Sections 11-12 will attempt Negative Binomial as an alternative.")
else:
    diagnostic_notes.append(
        f"OK Dispersion: median Var/Mean = {overall_dispersion:.2f} < {OVERDISPERSION_THRESHOLD}.")

if zero_inf_rate > ZERO_INFLATION_THRESHOLD:
    diagnostic_notes.append(
        f"WARNING Zero-inflation: {zero_inf_rate*100:.1f}% of groups have 0 changes "
        f"(threshold {ZERO_INFLATION_THRESHOLD*100:.0f}%). "
        f"Poisson/GLM estimates for sparse groups are upper-bound TTLs only.")
    POISSON_VIABLE = False
else:
    diagnostic_notes.append(
        f"OK Zero-inflation: {zero_inf_rate*100:.1f}% < {ZERO_INFLATION_THRESHOLD*100:.0f}%.")

for note in diagnostic_notes:
    print(note)

if not POISSON_VIABLE:
    print("\nPoisson diagnostics indicate significant issues.")
    print("Sections 11 (Poisson Process TTL) and 12 (GLM) will be skipped.")
else:
    print("\nPoisson diagnostics acceptable. Sections 11 and 12 will run.")

## Section 8 — Per-Key Volatility Summary

In [ ]:
key_stats = (
    price_ts.groupby("cache_key")
    .agg(
        n_obs             = ("amount_after_tax", "count"),
        n_distinct_prices = ("amount_after_tax", "nunique"),
        price_min         = ("amount_after_tax", "min"),
        price_max         = ("amount_after_tax", "max"),
        first_ts          = ("rq_timestamp",     "min"),
        chain_grouped     = ("chain_grouped",    "first"),
        rate_source       = ("rate_source",      "first"),
        horizon_bucket    = ("horizon_bucket",   "first"),
        duration_bucket   = ("duration_bucket",  "first"),
    )
    .reset_index()
)
key_stats["price_range"] = key_stats["price_max"] - key_stats["price_min"]
key_stats["volatile"]    = key_stats["n_distinct_prices"] > 1

print(f"Volatile keys: {key_stats.volatile.sum():,} / {len(key_stats):,} ({key_stats.volatile.mean()*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
key_stats["n_distinct_prices"].value_counts().sort_index().head(10).plot.bar(
    ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Distinct prices per cache key"); axes[0].tick_params(axis="x", rotation=0)

key_stats[key_stats.volatile & (key_stats.price_range < 500)]["price_range"]         .plot.hist(bins=40, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Price range per volatile key (<$500)")
axes[1].set_xlabel("Max - Min price (USD)")
plt.tight_layout(); plt.show()

## Section 9 — Kaplan-Meier Survival Analysis

We model **time-to-first-price-change** per cache key.

**Why KM not a simple mean?** Keys that never change in our 7-day window are **right-censored** (we know they lasted at least X hours, but we don't know when they would have changed). KM correctly accounts for this; a simple mean would be biased downward.

**Output:** For each horizon bucket, the TTL where P(still fresh) = TARGET_FRESHNESS.

In [ ]:
# ── Build survival dataset ────────────────────────────────────────────
first_obs    = price_ts.groupby("cache_key")["rq_timestamp"].min().reset_index(name="first_obs_ts")
first_change = (
    price_ts[price_ts["price_changed"]]
    .groupby("cache_key")["rq_timestamp"].min().reset_index(name="first_change_ts")
)
obs_end  = price_ts["rq_timestamp"].max()
survival = first_obs.merge(first_change, on="cache_key", how="left")
survival["event"]           = survival["first_change_ts"].notna().astype(int)
survival["first_change_ts"] = survival["first_change_ts"].fillna(obs_end)
survival["ttl_hours"]       = (
    (survival["first_change_ts"] - survival["first_obs_ts"]).dt.total_seconds() / 3600
).clip(lower=0)

survival = survival.merge(
    price_ts[["cache_key","horizon_bucket","duration_bucket","rate_source","chain_grouped"]]
    .drop_duplicates("cache_key"), on="cache_key", how="left")

print(f"Keys with observed change: {survival.event.sum():,} / {len(survival):,} ({survival.event.mean()*100:.1f}%)")
print(survival["ttl_hours"].describe())

In [ ]:
def km_ttl_at_freshness(kmf, target, min_ttl, max_ttl):
    """Largest t where KM survival >= target."""
    sf    = kmf.survival_function_.iloc[:,0]
    valid = sf[sf >= target]
    if len(valid) == 0: return min_ttl
    return float(np.clip(valid.index[-1], min_ttl, max_ttl))

# ── Overall KM ────────────────────────────────────────────────────────
kmf_all = KaplanMeierFitter()
kmf_all.fit(survival["ttl_hours"], survival["event"], label="All keys")
ttl_overall = km_ttl_at_freshness(kmf_all, TARGET_FRESHNESS, MIN_TTL, MAX_TTL)

fig, ax = plt.subplots(figsize=(12, 5))
kmf_all.plot_survival_function(ax=ax, ci_show=True)
for h, col in [(1,"red"),(4,"orange"),(12,"green"),(24,"purple")]:
    p = kmf_all.predict(h)
    ax.axvline(h, color=col, linestyle="--", lw=1, alpha=0.7)
    ax.text(h+0.3, p+0.02, f"{h}h: {p*100:.0f}%", color=col, fontsize=9)
ax.axhline(TARGET_FRESHNESS, color="navy", linestyle=":", lw=1.5,
           label=f"Target {TARGET_FRESHNESS*100:.0f}%")
ax.axvline(ttl_overall, color="navy", lw=2,
           label=f"TTL @ target = {ttl_overall:.1f}h")
ax.set_title("KM: Cache freshness survival (all keys)")
ax.set_xlabel("Hours since cached"); ax.set_ylabel("P(still fresh)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(); ax.set_xlim(0)
plt.tight_layout(); plt.show()

print(f"Implied TTL at {TARGET_FRESHNESS*100:.0f}% freshness (overall): {ttl_overall:.2f}h")
for h in [0.5,1,2,4,6,12,24,48]:
    print(f"  {h:4.1f}h: {kmf_all.predict(h)*100:.1f}% still fresh")

In [ ]:
# ── KM by horizon bucket + log-rank ───────────────────────────────────
palette_km = plt.cm.tab10.colors
buckets = [b for b in HORIZON_LABELS
           if survival[survival["horizon_bucket"]==b]["event"].sum() >= MIN_EVENTS_PER_KM_GROUP]
print(f"Horizon buckets with >= {MIN_EVENTS_PER_KM_GROUP} events: {buckets}")

fig, ax = plt.subplots(figsize=(13, 6))
kmf_dict  = {}
ttl_rows  = []
for i, bucket in enumerate(buckets):
    grp   = survival[survival["horizon_bucket"] == bucket]
    kmf_b = KaplanMeierFitter()
    kmf_b.fit(grp["ttl_hours"], grp["event"], label=bucket)
    kmf_b.plot_survival_function(ax=ax, ci_show=False, color=palette_km[i % 10])
    kmf_dict[bucket] = kmf_b
    ttl_b = km_ttl_at_freshness(kmf_b, TARGET_FRESHNESS, MIN_TTL, MAX_TTL)
    ttl_rows.append({"horizon_bucket": bucket, "n_keys": len(grp),
                     "n_events": int(grp["event"].sum()),
                     f"TTL_at_{int(TARGET_FRESHNESS*100)}pct_h": round(ttl_b, 2)})

ax.axhline(TARGET_FRESHNESS, color="black", linestyle=":", lw=1.5,
           label=f"{TARGET_FRESHNESS*100:.0f}% target")
ax.set_title("KM Survival by horizon bucket")
ax.set_xlabel("Hours since cached"); ax.set_ylabel("P(still fresh)")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(loc="upper right", fontsize=9); ax.set_xlim(0)
plt.tight_layout(); plt.show()

ttl_lookup_km = pd.DataFrame(ttl_rows)
ttl_col = [c for c in ttl_lookup_km.columns if "TTL" in c][0]
print("\n=== KM-derived TTL lookup ===")
print(ttl_lookup_km.to_string(index=False))

In [ ]:
# ── Log-rank tests between adjacent horizon buckets ───────────────────
print("Log-rank tests between adjacent horizon buckets:")
print(f"{'Bucket A':<18} {'Bucket B':<18} {'p-value':>10}  Significant?")
print("-" * 55)
for i in range(len(buckets) - 1):
    g1  = survival[survival["horizon_bucket"] == buckets[i]]
    g2  = survival[survival["horizon_bucket"] == buckets[i+1]]
    res = logrank_test(g1["ttl_hours"], g2["ttl_hours"],
                       event_observed_A=g1["event"], event_observed_B=g2["event"])
    sig = "YES" if res.p_value < LOGRANK_SIGNIFICANCE else "no"
    print(f"{buckets[i]:<18} {buckets[i+1]:<18} {res.p_value:>10.4f}  {sig}")

In [ ]:
# ── Log-rank tests across other dimensions (exploratory) ──────────────
print("=== Which other dimensions significantly differentiate survival? ===")
for dim in ["rate_source", "duration_bucket", "chain_grouped"]:
    grps = [g for g in survival[dim].dropna().unique()
            if survival[survival[dim]==g]["event"].sum() >= MIN_EVENTS_PER_KM_GROUP]
    if len(grps) < 2:
        print(f"{dim:<20}: not enough groups with sufficient events"); continue
    sub = survival[survival[dim].isin(grps)]
    try:
        res = multivariate_logrank_test(sub["ttl_hours"], sub[dim], sub["event"])
        sig = "SIGNIFICANT" if res.p_value < LOGRANK_SIGNIFICANCE else "not significant"
        print(f"{dim:<20}: p = {res.p_value:.4f}  -> {sig}")
    except Exception as e:
        print(f"{dim:<20}: test failed - {e}")

In [ ]:
# ── KM by rate_source ─────────────────────────────────────────────────
rs_with_data = [rs for rs in survival["rate_source"].dropna().unique()
                if survival[survival["rate_source"]==rs]["event"].sum() >= MIN_EVENTS_PER_KM_GROUP]

if len(rs_with_data) >= 2:
    fig, ax = plt.subplots(figsize=(12, 5))
    for i, rs in enumerate(rs_with_data):
        grp   = survival[survival["rate_source"] == rs]
        kmf_rs = KaplanMeierFitter()
        kmf_rs.fit(grp["ttl_hours"], grp["event"], label=f"Rate source {rs}")
        kmf_rs.plot_survival_function(ax=ax, ci_show=False, color=palette_km[i])
    ax.axhline(TARGET_FRESHNESS, color="black", linestyle=":", lw=1.5)
    ax.set_title("KM Survival by rate_source")
    ax.set_xlabel("Hours"); ax.set_ylabel("P(still fresh)")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(); ax.set_xlim(0)
    plt.tight_layout(); plt.show()
else:
    print("Not enough rate_source groups with sufficient events for KM comparison.")

## Section 10 — Fixed TTL Grid Search

Find the single best fixed TTL by simulating staleness for each value in `TTL_GRID`. Gives us a data-driven optimal fixed TTL rather than an arbitrary choice.

In [ ]:
# ── Sample cache keys for simulation ─────────────────────────────────
# We sample complete key histories (not random rows) to preserve time-series integrity.
np.random.seed(42)
key_size_map   = price_ts.groupby("cache_key").size()
all_keys       = key_size_map.index.values
shuffled_idx   = np.random.permutation(len(all_keys))
cumulative     = np.cumsum(key_size_map.values[shuffled_idx])
n_keys_to_take = int(np.searchsorted(cumulative, SIMULATION_SAMPLE_ROWS)) + 1
sim_keys       = all_keys[shuffled_idx[:n_keys_to_take]]
sim_df         = price_ts[price_ts["cache_key"].isin(sim_keys)].copy()
print(f"Simulation dataset: {len(sim_keys):,} keys, {len(sim_df):,} rows")

In [ ]:
# ── Core simulation engine ─────────────────────────────────────────────
def simulate_staleness(df_sim, get_ttl_fn, strategy_name, verbose=True):
    """
    Replay price time-series per cache key.
    At each observation: if elapsed >= TTL -> refresh cache.
    Returns dict with three staleness metrics:
      stale_pct           : % observations where relative error exceeds STALENESS_THRESHOLD
      avg_abs_err_stale   : mean absolute price error when stale ($)
      avg_pct_err_stale   : mean relative price error when stale (%)
    """
    stale_count = total_count = 0
    abs_err_sum = pct_err_sum = 0.0

    for key, group in df_sim.groupby("cache_key"):
        rows = list(group.sort_values("rq_timestamp").itertuples(index=False))
        if len(rows) < 2:
            continue

        r0           = rows[0]
        cache_price  = r0.amount_after_tax
        last_refresh = r0.rq_timestamp
        ttl          = get_ttl_fn(r0)

        for row in rows[1:]:
            true_price = row.amount_after_tax
            if pd.isna(true_price):
                continue

            hours_elapsed = (row.rq_timestamp - last_refresh).total_seconds() / 3600
            if hours_elapsed >= ttl:
                cache_price  = true_price
                last_refresh = row.rq_timestamp
                ttl          = get_ttl_fn(row)

            is_stale = (
                (not pd.isna(cache_price)) and
                (cache_price > 0) and
                (abs(cache_price - true_price) / cache_price > STALENESS_THRESHOLD)
            )

            total_count += 1
            if is_stale:
                stale_count += 1
                abs_err = abs(cache_price - true_price)
                abs_err_sum += abs_err
                pct_err_sum += abs_err / true_price * 100

    stale_pct   = stale_count / total_count * 100 if total_count else 0
    avg_abs_err = abs_err_sum / stale_count if stale_count else 0
    avg_pct_err = pct_err_sum / stale_count if stale_count else 0

    result = {
        "strategy"              : strategy_name,
        "stale_pct"             : round(stale_pct, 2),
        "avg_abs_err_when_stale": round(avg_abs_err, 2),
        "avg_pct_err_when_stale": round(avg_pct_err, 2),
        "total_obs"             : total_count,
        "stale_obs"             : stale_count,
    }
    if verbose:
        print(f"[{strategy_name:<45}] {stale_pct:6.2f}% stale "
              f"| avg $err={avg_abs_err:.2f}  avg %err={avg_pct_err:.1f}%")
    return result

print("Simulation engine ready.")


In [ ]:
# ── Grid search with two-sided cache-size constraints ──────────────────
def estimate_cache_size_mb(df_sim, ttl_hours, n_samples=200):
    """Estimate cache memory by 95th percentile active key count over sampled timestamps."""
    ts = df_sim["rq_timestamp"].dropna().sort_values().unique()
    if len(ts) == 0:
        return 0.0

    sample_n = min(n_samples, len(ts))
    idx = np.linspace(0, len(ts) - 1, sample_n).astype(int)
    sampled_ts = pd.to_datetime(ts[idx])

    active_counts = []
    for t in sampled_ts:
        window_start = t - pd.Timedelta(hours=float(ttl_hours))
        mask = (df_sim["rq_timestamp"] >= window_start) & (df_sim["rq_timestamp"] <= t)
        active_counts.append(df_sim.loc[mask, "cache_key"].nunique())

    p95_count = np.percentile(active_counts, 95) if active_counts else 0
    return float(p95_count * AVG_ENTRY_SIZE_BYTES / 1e6)

print("Running fixed-TTL grid search...")
grid_results = []
for ttl_val in TTL_GRID:
    res = simulate_staleness(sim_df, lambda r, t=ttl_val: t, f"Fixed TTL {ttl_val}h", verbose=False)
    est_mb = estimate_cache_size_mb(sim_df, ttl_val)
    feasible = (est_mb >= MIN_CACHE_UTIL_MB) and (est_mb <= MAX_CACHE_SIZE_MB)

    res["ttl_value"] = ttl_val
    res["estimated_cache_mb"] = round(est_mb, 2)
    res["feasible"] = feasible
    grid_results.append(res)
    print(f"  TTL={ttl_val:5.2f}h  stale={res['stale_pct']:.2f}%  est_cache={est_mb:.2f}MB  feasible={feasible}")

grid_df = pd.DataFrame(grid_results)

feasible_df = grid_df[grid_df["feasible"]]
selection_mode = "two-sided"
if len(feasible_df) > 0:
    best_row = feasible_df.loc[feasible_df["stale_pct"].idxmin()]
else:
    print("WARNING: No TTL satisfies both constraints.")
    print("Relaxing utilization floor. Applying memory cap only.")
    mem_ok = grid_df[grid_df["estimated_cache_mb"] <= MAX_CACHE_SIZE_MB]
    selection_mode = "memory-only"
    if len(mem_ok) > 0:
        best_row = mem_ok.loc[mem_ok["stale_pct"].idxmin()]
    else:
        print("WARNING: All TTLs exceed memory cap. Selecting smallest TTL.")
        selection_mode = "min-ttl-fallback"
        best_row = grid_df.loc[grid_df["ttl_value"].idxmin()]

OPTIMAL_FIXED_TTL = best_row["ttl_value"]
grid_df["selected"] = False
grid_df.loc[best_row.name, "selected"] = True

# Constraint binding summary
tol = 0.25
if selection_mode == "two-sided":
    if abs(best_row["estimated_cache_mb"] - MAX_CACHE_SIZE_MB) <= tol:
        binding_constraint = "upper"
    elif abs(best_row["estimated_cache_mb"] - MIN_CACHE_UTIL_MB) <= tol:
        binding_constraint = "lower"
    else:
        binding_constraint = "neither"
else:
    binding_constraint = selection_mode

print()
print("=== TTL GRID SUMMARY ===")
print(grid_df[["ttl_value", "stale_pct", "estimated_cache_mb", "feasible", "selected"]].to_string(index=False))
print()
print(f"Feasible TTL count: {len(feasible_df)} / {len(grid_df)}")
print(
    f"Selected TTL: {OPTIMAL_FIXED_TTL}h | "
    f"stale={best_row['stale_pct']:.2f}% | cache={best_row['estimated_cache_mb']:.2f}MB | "
    f"binding={binding_constraint}"
)

# Dual-axis plot with feasible region
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(grid_df["ttl_value"], grid_df["stale_pct"], "o-", color="steelblue", lw=2, ms=5, label="stale_pct")
ax2.plot(grid_df["ttl_value"], grid_df["estimated_cache_mb"], "o-", color="gray", lw=2, ms=4, label="estimated_cache_mb")

ax2.axhline(MAX_CACHE_SIZE_MB, color="red", linestyle="--", lw=1.5, label=f"MAX {MAX_CACHE_SIZE_MB}MB")
ax2.axhline(MIN_CACHE_UTIL_MB, color="orange", linestyle="--", lw=1.5, label=f"MIN util {MIN_CACHE_UTIL_MB:.1f}MB")
ax2.axhspan(MIN_CACHE_UTIL_MB, MAX_CACHE_SIZE_MB, color="lightgreen", alpha=0.2)

ax1.axvline(OPTIMAL_FIXED_TTL, color="purple", lw=2, linestyle=":", label=f"Optimal {OPTIMAL_FIXED_TTL}h")

ax1.set_title("Fixed TTL grid search with two-sided cache-size feasibility")
ax1.set_xlabel("TTL (hours)")
ax1.set_ylabel("% stale observations", color="steelblue")
ax2.set_ylabel("Estimated cache size (MB)", color="gray")

legend_h1, legend_l1 = ax1.get_legend_handles_labels()
legend_h2, legend_l2 = ax2.get_legend_handles_labels()
ax1.legend(legend_h1 + legend_h2, legend_l1 + legend_l2, loc="best")
plt.tight_layout(); plt.show()


## Section 11 — Poisson Process TTL

**Concept:** Model price change events as a Poisson process with rate lambda (changes/hour) per group.

- P(no change in time t) = exp(-lambda * t)
- For target freshness p: **TTL = -ln(p) / lambda**

lambda estimated as `total_changes / total_observation_hours` per group `(chain, rate_source, horizon_bucket)`.
This correctly handles different window lengths: 1 change in 168h -> lambda=0.006/h vs 1 change in 24h -> lambda=0.042/h.

*Conditional on diagnostics from Section 7.*

In [ ]:
if not POISSON_VIABLE:
    print("Poisson diagnostics did not pass (see Section 7 conclusion).")
    print("Data shows significant zero-inflation that violates Poisson assumptions.")
    print("This section is SKIPPED. Strategy 3 (Poisson TTL) excluded from comparison.")
    INCLUDE_POISSON_STRATEGY = False
else:
    INCLUDE_POISSON_STRATEGY = True
    global_lambda = group_agg[group_agg["total_obs_h"] > 0].eval(
        "total_changes / total_obs_h").median()
    group_agg["lambda_per_h"] = (
        group_agg["total_changes"] / group_agg["total_obs_h"].replace(0, np.nan))
    lambda_lookup = (
        group_agg.set_index(["chain_grouped","rate_source","horizon_bucket"])
        ["lambda_per_h"].to_dict()
    )
    print(f"Global median lambda : {global_lambda:.5f} changes/hour")
    print(f"Implied TTL at target: {-np.log(TARGET_FRESHNESS)/global_lambda:.1f}h")

    hm_data = (
        group_agg[group_agg["chain_grouped"] != "Other"]
        .pivot_table(index="chain_grouped", columns="horizon_bucket",
                     values="lambda_per_h", aggfunc="mean")
    )
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(hm_data, annot=True, fmt=".4f", cmap="YlOrRd", ax=ax,
                cbar_kws={"label": "lambda (changes/hour)"})
    ax.set_title("Lambda heatmap - chain x horizon bucket")
    plt.tight_layout(); plt.show()

    group_agg["implied_ttl"] = (
        -np.log(TARGET_FRESHNESS) / group_agg["lambda_per_h"]
    ).clip(MIN_TTL, MAX_TTL)
    group_agg["implied_ttl"].plot.hist(bins=40, color="steelblue", edgecolor="white")
    plt.title(f"Implied TTL distribution at {TARGET_FRESHNESS*100:.0f}% freshness")
    plt.xlabel("TTL (hours)"); plt.tight_layout(); plt.show()

## Section 12 — Poisson Regression GLM

**Concept:** Model lambda as a function of features:

```
log(lambda) = b0 + b1*horizon + b2*rate_source + b3*chain + b4*duration + log(t)
```

The `log(t)` term is an **offset** (observation window in hours) that normalises for different exposure lengths.

Exponentiated coefficients = **Incidence Rate Ratios (IRR)**: "this feature multiplies the change rate by X".

*Conditional on diagnostics from Section 7.*

In [ ]:
INCLUDE_GLM_STRATEGY = False  # default; overridden below if viable

if not POISSON_VIABLE:
    print("Poisson diagnostics did not pass (see Section 7).")
    print("Data does not meet basic assumptions for Poisson Regression GLM.")
    print("This section is SKIPPED. Strategy 4 (GLM) excluded from comparison.")
else:
    glm_data = group_agg[
        (group_agg["total_obs_h"] > 0) & (group_agg["total_obs"] >= 5)
    ].copy()
    glm_data["log_obs_h"]     = np.log(glm_data["total_obs_h"].clip(lower=0.01))
    glm_data["n_changes_int"] = glm_data["total_changes"].astype(int)

    def bucket_midpoint(label):
        try:
            parts = str(label).replace("+","").split("-")
            return (int(parts[0]) + int(parts[-1])) / 2
        except: return np.nan

    glm_data["horizon_mid"] = glm_data["horizon_bucket"].apply(bucket_midpoint)
    print(f"GLM dataset: {glm_data.shape[0]} rows")
    print(glm_data["n_changes_int"].describe())

    use_negbin = (overall_dispersion > OVERDISPERSION_THRESHOLD)
    family     = sm.families.NegativeBinomial() if use_negbin else sm.families.Poisson()
    mtype      = "Negative Binomial" if use_negbin else "Poisson"
    print(f"\nFitting {mtype} GLM...")

    try:
        model = smf.glm(
            formula="n_changes_int ~ horizon_mid + C(rate_source) + C(chain_grouped)",
            data=glm_data, family=family, offset=glm_data["log_obs_h"]
        ).fit()
        print(model.summary())

        irr = pd.DataFrame({"coef": model.params, "IRR": np.exp(model.params), "p": model.pvalues}).round(4)
        print("\n=== IRR (exp(coef)) - IRR>1 increases rate, IRR<1 decreases rate ===")
        print(irr.to_string())

        glm_data["lambda_glm"] = np.exp(model.fittedvalues) / glm_data["total_obs_h"]
        glm_data["implied_ttl_glm"] = (
            -np.log(TARGET_FRESHNESS) / glm_data["lambda_glm"]
        ).clip(MIN_TTL, MAX_TTL)

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(glm_data["n_changes_int"], model.fittedvalues, alpha=0.4, s=15)
        mx = max(glm_data["n_changes_int"].max(), model.fittedvalues.max())
        ax.plot([0,mx],[0,mx],"r--",lw=1.5,label="y=x")
        ax.set_title(f"{mtype} GLM: Observed vs Fitted"); ax.legend()
        plt.tight_layout(); plt.show()

        glm_lambda_lookup = glm_data.set_index(
            ["chain_grouped","rate_source","horizon_bucket"])["lambda_glm"].to_dict()
        INCLUDE_GLM_STRATEGY = True

    except Exception as ex:
        print(f"GLM fitting failed: {ex}")

## Section 13 — Strategy Simulation

Compare all strategies head-to-head on the sampled real data.

| # | Strategy | Description |
|---|---|---|
| 0 | Baseline | Never refresh |
| 1 | Fixed TTL (optimal) | Best from grid search |
| 2 | KM-Adaptive | Data-driven TTL per horizon bucket |
| 3 | Poisson TTL | TTL = -ln(target)/lambda per group |
| 4 | GLM TTL | TTL = -ln(target)/lambda from GLM |

In [ ]:

# ── TTL functions ──────────────────────────────────────────────────────
def fixed_ttl_optimal(row):
    return float(OPTIMAL_FIXED_TTL)

_km_ttl_col = [c for c in ttl_lookup_km.columns if "TTL" in c][0]
_km_ttl_map = ttl_lookup_km.set_index("horizon_bucket")[_km_ttl_col].to_dict()


def _normalized_horizon_days(row):
    """Timezone-safe horizon calculation for both real and synthetic rows."""
    ts = pd.Timestamp(row.rq_timestamp)
    stay = pd.Timestamp(row.rq_stay_start_date)

    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)
    if stay.tzinfo is not None:
        stay = stay.tz_convert(None)

    return max(0, (stay.normalize() - ts.normalize()).days)


def km_adaptive_ttl(row):
    horizon = _normalized_horizon_days(row)
    bucket = assign_bucket(int(horizon), HORIZON_BREAKS, HORIZON_LABELS)
    return float(np.clip(_km_ttl_map.get(bucket, ttl_overall), MIN_TTL, MAX_TTL))


def poisson_ttl(row):
    horizon = _normalized_horizon_days(row)
    bucket = assign_bucket(int(horizon), HORIZON_BREAKS, HORIZON_LABELS)
    lam = lambda_lookup.get((row.chain_grouped, row.rate_source, bucket), global_lambda)
    if pd.isna(lam) or lam <= 0:
        lam = global_lambda
    return float(np.clip(-np.log(TARGET_FRESHNESS) / lam, MIN_TTL, MAX_TTL))


def glm_ttl(row):
    horizon = _normalized_horizon_days(row)
    bucket = assign_bucket(int(horizon), HORIZON_BREAKS, HORIZON_LABELS)
    lam = glm_lambda_lookup.get((row.chain_grouped, row.rate_source, bucket), global_lambda)
    if pd.isna(lam) or lam <= 0:
        lam = global_lambda
    return float(np.clip(-np.log(TARGET_FRESHNESS) / lam, MIN_TTL, MAX_TTL))


print("TTL functions defined.")


In [ ]:
print("Running strategies on REAL data...\n")
sim_results = []
sim_results.append(simulate_staleness(sim_df, lambda r: 1e9,      "Baseline (no refresh)"))
sim_results.append(simulate_staleness(sim_df, fixed_ttl_optimal,   f"Fixed TTL {OPTIMAL_FIXED_TTL}h"))
sim_results.append(simulate_staleness(sim_df, km_adaptive_ttl,     "KM-Adaptive TTL"))
if INCLUDE_POISSON_STRATEGY:
    sim_results.append(simulate_staleness(sim_df, poisson_ttl,     "Poisson Process TTL"))
if INCLUDE_GLM_STRATEGY:
    sim_results.append(simulate_staleness(sim_df, glm_ttl,         "Poisson GLM TTL"))

results_df = pd.DataFrame(sim_results)
results_df["reduction_vs_baseline"] = (
    (results_df.loc[0,"stale_pct"] - results_df["stale_pct"]) /
     results_df.loc[0,"stale_pct"] * 100
).round(1)

print("\n=== STRATEGY COMPARISON TABLE ===")
print(results_df[["strategy","stale_pct","avg_abs_err_when_stale",
                   "avg_pct_err_when_stale","reduction_vs_baseline"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
palette_s = ["#d62728","#ff7f0e","#1f77b4","#2ca02c","#9467bd"]
short_names = [s.split("(")[0].strip()[:22] for s in results_df["strategy"]]

for ax, col, lbl in [
    (axes[0], "stale_pct",               "% Stale"),
    (axes[1], "avg_abs_err_when_stale",  "Avg $err when stale"),
    (axes[2], "avg_pct_err_when_stale",  "Avg %err when stale"),
]:
    bars = ax.bar(short_names, results_df[col],
                  color=palette_s[:len(results_df)], edgecolor="white")
    for b, v in zip(bars, results_df[col]):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()*1.02,
                f"{v:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_title(lbl); ax.tick_params(axis="x", rotation=20)
    ax.set_xticks(range(len(short_names))); ax.set_xticklabels(short_names, rotation=25, ha="right")

plt.suptitle("Strategy Comparison - Real Data", fontsize=13)
plt.tight_layout(); plt.show()

## Section 14 — Synthetic Data Validation

Generate 2 extra weeks of synthetic data using empirical Poisson rates, then re-run all strategies. Consistent rankings between real and synthetic confirm results are robust.

In [ ]:
def generate_synthetic(n_keys=2000, n_days=14, seed=0):
    rng      = np.random.default_rng(seed)
    start_ts = price_ts["rq_timestamp"].max() + pd.Timedelta(hours=1)
    gaps_h   = price_ts["obs_gap_h"].dropna()
    med_gap  = float(gaps_h.median()); std_gap = float(gaps_h.std())
    base     = key_stats.sample(n=n_keys, replace=True, random_state=seed).reset_index(drop=True)

    records = []
    for _, row in base.iterrows():
        chain, rs, hb = row["chain_grouped"], row["rate_source"], row["horizon_bucket"]
        lam = lambda_lookup.get((chain, rs, hb), global_lambda)
        if pd.isna(lam) or lam <= 0:
            lam = global_lambda
        syn_key = f"SYN|{row['cache_key']}|{rng.integers(int(1e7))}"
        price   = float(max(10, rng.normal(row["price_min"] + row["price_range"] / 2,
                                           max(1, row["price_range"] * 0.15))))
        t       = start_ts + pd.Timedelta(hours=float(rng.uniform(0, 2)))
        end_t   = start_ts + pd.Timedelta(days=n_days)
        stay_start = start_ts + pd.Timedelta(days=int(rng.integers(30, 120)))
        while t < end_t:
            records.append({"cache_key": syn_key, "rq_timestamp": t,
                            "amount_after_tax": round(price, 2),
                            "chain_grouped": chain, "rate_source": rs,
                            "rq_stay_start_date": stay_start,
                            "duration_nights": max(1, int(row["price_range"] // 100 or 2))})
            gap_h = max(0.1, float(rng.normal(med_gap, std_gap)))
            t += pd.Timedelta(hours=gap_h)
            if rng.poisson(lam * gap_h) > 0:
                price = max(10, price * (1 + float(rng.normal(0, STALENESS_THRESHOLD * 2))))

    syn = pd.DataFrame(records)
    syn["prev_price"] = syn.groupby("cache_key")["amount_after_tax"].shift(1)
    syn["price_changed"] = (
        syn["prev_price"].notna() &
        (syn["prev_price"] > 0) &
        (abs(syn["amount_after_tax"] - syn["prev_price"]) / syn["prev_price"] > STALENESS_THRESHOLD)
    )
    syn["horizon_days"] = 60
    syn["horizon_bucket"] = syn["horizon_days"].apply(
        lambda x: assign_bucket(int(x), HORIZON_BREAKS, HORIZON_LABELS)
    )
    print(f"Synthetic: {len(syn):,} rows, {syn.cache_key.nunique():,} keys")
    return syn

syn_df = generate_synthetic(n_keys=2000, n_days=14) if INCLUDE_POISSON_STRATEGY else None


In [ ]:
if syn_df is None:
    print("Synthetic simulation skipped (Poisson lambda not available).")
else:
    print("Running strategies on SYNTHETIC data...\n")
    syn_results = []
    syn_results.append(simulate_staleness(syn_df, lambda r: 1e9,     "Baseline"))
    syn_results.append(simulate_staleness(syn_df, fixed_ttl_optimal,  f"Fixed TTL {OPTIMAL_FIXED_TTL}h"))
    syn_results.append(simulate_staleness(syn_df, km_adaptive_ttl,    "KM-Adaptive"))
    if INCLUDE_POISSON_STRATEGY:
        syn_results.append(simulate_staleness(syn_df, poisson_ttl,    "Poisson TTL"))
    if INCLUDE_GLM_STRATEGY:
        syn_results.append(simulate_staleness(syn_df, glm_ttl,        "GLM TTL"))

    syn_df_r = pd.DataFrame(syn_results)
    names_s  = [s.split("(")[0].strip()[:22] for s in results_df["strategy"]]
    x = np.arange(len(names_s)); w = 0.35
    fig, ax = plt.subplots(figsize=(13, 5))
    b1 = ax.bar(x-w/2, results_df["stale_pct"],  w, label="Real",      color="steelblue", edgecolor="white")
    b2 = ax.bar(x+w/2, syn_df_r["stale_pct"],    w, label="Synthetic", color="coral",     edgecolor="white")
    for b in list(b1)+list(b2):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
                f"{b.get_height():.1f}%", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(names_s, rotation=20, ha="right")
    ax.set_title("% Stale - Real vs Synthetic"); ax.set_ylabel("% Stale"); ax.legend()
    plt.tight_layout(); plt.show()

## Section 15 — Presentation Summary Figures

Clean, labeled, export-quality figures.

In [ ]:
# Figure 1: Motivation - staleness example ────────────────────────────
volatile_keys = key_stats[key_stats["volatile"] & (key_stats["n_obs"] >= 5)]
demo_keys     = volatile_keys.sample(3, random_state=7)["cache_key"].values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key in zip(axes, demo_keys):
    grp = price_ts[price_ts["cache_key"] == key].sort_values("rq_timestamp")
    ax.step(grp["rq_timestamp"].values, grp["amount_after_tax"].values, where="post", lw=2, color="steelblue")
    ax.scatter(grp["rq_timestamp"].values, grp["amount_after_tax"].values, s=30, color="steelblue", zorder=5)
    cp = grp["amount_after_tax"].values[0]
    times = grp["rq_timestamp"].values
    prices = grp["amount_after_tax"].values
    for i in range(1, len(prices)):
        if cp > 0 and abs(prices[i] - cp) / cp > STALENESS_THRESHOLD:
            ax.axvspan(times[i - 1], times[i], alpha=0.15, color="red")
            cp = prices[i]
    ax.set_title(f"Key: ...{key[-18:]}", fontsize=8)
    ax.set_xlabel("Time")
    ax.set_ylabel("Price (USD)")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, fontsize=7)

plt.suptitle(
    "Figure 1: Price-over-time for 3 volatile cache keys\n"
    "Red shading = stale window (relative price change > threshold)",
    fontsize=11,
)
plt.tight_layout(); plt.show()


In [ ]:
# Figure 2: Hypothesis 2x2 panel ──────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].bar(h1["horizon_days_int"], h1["change_rate_pct"],
              color="steelblue", width=1, edgecolor="none", alpha=0.7)
axes[0,0].set_title("H1: Change rate by booking horizon")
axes[0,0].set_xlabel("Days until stay"); axes[0,0].set_ylabel("Change rate (%)")
axes[0,0].set_xlim(-1, 181)

axes[0,1].bar(h2["duration_int"].astype(str), h2["change_rate_pct"],
              color="coral", edgecolor="white")
axes[0,1].set_title("H2: Change rate by stay duration")
axes[0,1].set_xlabel("Duration (nights)")

if len(h3_rate) > 0:
    axes[1,0].bar(h3_rate["stay_start_month"].map(month_names), h3_rate["change_rate_pct"],
                  color="purple", edgecolor="white")
    axes[1,0].set_title("H3: Change rate by stay start month")
    axes[1,0].tick_params(axis="x", rotation=30)

axes[1,1].bar(h4_rate["rate_source"].astype(str), h4_rate["change_rate_pct"],
              color="green", edgecolor="white")
axes[1,1].set_title("H4: Change rate by rate source")
axes[1,1].set_xlabel("Rate source")

plt.suptitle("Figure 2: What drives price volatility?", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 3: KM curves + TTL table ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for i, (bucket, kmf_b) in enumerate(kmf_dict.items()):
    kmf_b.plot_survival_function(ax=axes[0], ci_show=False, color=palette_km[i % 10])
axes[0].axhline(TARGET_FRESHNESS, color="black", linestyle=":", lw=1.5,
                label=f"{TARGET_FRESHNESS*100:.0f}% freshness target")
axes[0].set_title("KM Survival by horizon bucket")
axes[0].set_xlabel("Hours since cached"); axes[0].set_ylabel("P(still fresh)")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].legend(fontsize=9); axes[0].set_xlim(0)

axes[1].axis("off")
table_vals = [[row["horizon_bucket"], str(row["n_events"]), f"{row[_km_ttl_col]}h"]
              for _, row in ttl_lookup_km.iterrows()]
tbl = axes[1].table(cellText=table_vals,
                    colLabels=["Horizon bucket","# change events",
                               f"TTL @ {TARGET_FRESHNESS*100:.0f}%"],
                    cellLoc="center", loc="center", bbox=[0,0,1,1])
tbl.auto_set_font_size(False); tbl.set_fontsize(11)
axes[1].set_title(f"Data-driven TTL per bucket (target: {TARGET_FRESHNESS*100:.0f}% fresh)", pad=20)

plt.suptitle("Figure 3: Kaplan-Meier freshness curves & derived TTL lookup", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Figure 4: Final strategy comparison ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
names_f = [s.split("(")[0].strip() for s in results_df["strategy"]]
cols_f  = ["#d62728","#ff7f0e","#1f77b4","#2ca02c","#9467bd"]

for ax, col, lbl in [(axes[0],"stale_pct","% Stale"),
                      (axes[1],"reduction_vs_baseline","% Reduction vs Baseline")]:
    bars = ax.bar(names_f, results_df[col], color=cols_f[:len(results_df)], edgecolor="white")
    for b, v in zip(bars, results_df[col]):
        ax.text(b.get_x()+b.get_width()/2, max(b.get_height(),0)+0.3,
                f"{v:.1f}", ha="center", va="bottom", fontweight="bold", fontsize=10)
    ax.set_title(lbl); ax.tick_params(axis="x", rotation=20)
    ax.set_xticks(range(len(names_f))); ax.set_xticklabels(names_f, rotation=25, ha="right")

plt.suptitle("Figure 4: Final Strategy Comparison", fontsize=14)
plt.tight_layout(); plt.show()

print("\n=== FINAL RESULTS TABLE ===")
print(results_df[["strategy","stale_pct","avg_abs_err_when_stale",
                   "avg_pct_err_when_stale","reduction_vs_baseline"]]
      .rename(columns={"stale_pct":"% Stale","avg_abs_err_when_stale":"Avg $err (stale)",
                        "avg_pct_err_when_stale":"Avg %err (stale)",
                        "reduction_vs_baseline":"% Reduction vs Baseline"})
      .to_string(index=False))